In [ ]:
import hydromt
import xarray as xr
import numpy as np
import pandas as pd
from os.path import join, basename
import glob
import geopandas as gpd
import pygeos
import matplotlib.pyplot as plt

In [ ]:
bbox =  32.0, -21.5, 35.5, -17.0
root = r'/mnt/bazis/projects/hydromt-floodmodelling/00_data/'
chunks={'stations': 100, 'time':-1}
rm = {'station_x_coordinate':'lon', 'station_y_coordinate':'lat'}

In [ ]:
## read AVISO MDT
mdt_fn = r'/mnt/bazis_data/hydrology/topography/mdt/aviso/MDT_CNES_CLS18_global_filled.tif'
da_mdt = hydromt.open_raster(mdt_fn)
da_mdt

In [ ]:
## read CMF pits
mapdir = r'/mnt/bazis/projects/hydromt-floodmodelling/02_models/cmf/map/beira_06min'
gdf_pits = gpd.read_file(join(mapdir, 'gis', 'pits.geojson'))

In [ ]:
def cmf_pit_mapping(gdf_gtsm, name, gdf_pits=gdf_pits, mapdir=mapdir, max_dist=10e3):
    pts = pygeos.points([g.coords[:][0] for g in gdf_pits.geometry])
    gtsm_idx = gdf_gtsm.sindex.nearest(pts)[1]
    gdf_pits['gtsm_idx'] = gtsm_idx + 1
    gdf_pits['gtsm_dst']  = gdf_gtsm.iloc[gtsm_idx].to_crs(32736).distance(gdf_pits.to_crs(32736), align=False).values
    gdf_pits = gdf_pits[gdf_pits['gtsm_dst']<max_dist]
    nlinks = gdf_pits.index.size
    print(nlinks)
    cmf_gtsm_ref = gdf_pits[['col', 'row', 'gtsm_idx']].fillna(0).values.astype(np.int32)
    with open(join(mapdir, f'{name}.txt'), 'w') as f:
        f.write(f'{nlinks:d}\n')
        np.savetxt(
            f, 
            cmf_gtsm_ref,
            fmt='%3.d',
        )
    return gdf_pits, cmf_gtsm_ref

In [ ]:
## CODEC
name = 'GTSM_CODEC'
fns = glob.glob(join(root, name, 'reanalysis_waterlevel_10min_*_v1_beira.nc'))
ds_gtsm0 = xr.open_dataset(fns[0], chunks=chunks).rename(rm).vector.clip_bbox(bbox)
gdf_gtsm = ds_gtsm0.vector.to_gdf().set_crs(4326)
gdf_gtsm

In [ ]:
gdf_pits1, cmf_gtsm_ref = cmf_pit_mapping(gdf_gtsm, name)
fig, ax = plt.subplots(1,1, figsize=(6,6))
gdf_pits.plot(markersize=(gdf_pits['upa_10'].values-7)*7, color='b', ax=ax)
gdf_pits1.plot(markersize=(gdf_pits1['upa_10'].values-7)*7, color='k', ax=ax)
gdf_gtsm.iloc[np.unique(gdf_pits1['gtsm_idx'].values-1)].plot(markersize=15, color='r', marker='^', ax=ax)
cmf_gtsm_ref

In [ ]:
mdt0 = da_mdt.raster.sample(gdf_gtsm).rename({'index': 'stations'}).reset_coords(drop=True)
for fn in fns: 
    print(basename(fn))
    fn_out = fn.replace('.nc', '_egm.nc')
    ds_gtsm = xr.open_dataset(fn, chunks=chunks).sel(stations=gdf_gtsm.index.values)
    ds_gtsm = ds_gtsm + mdt0
    encoding = {'waterlevel': {'dtype': 'float32'}}
    ds_gtsm.to_netcdf(fn_out, encoding=encoding)

In [ ]:
## IDAI / ELOISE
name = 'GTSM_20190218_20190401_IDAI'
tstart, tstop = '2019-01-01', '2019-05-01'
prefix = 'idai'

name = 'GTSM_20201218_20210201_ELOISE'
tstart, tstop = '2021-01-01', '2021-03-01'
prefix = 'eloise'

var = 'waterlevel'
runs = ['era5', 'spw', 'tides', 'era5_tides', 'spw_tides', 'era5_spw_tides']
fn0 = join(root, name, f'{prefix}_{runs[0]}_his.nc')
da_gtsm0 = xr.open_dataset(fn0, chunks=chunks).rename(rm)[var]
da_gtsm0['stations'] = da_gtsm0.stations
da_gtsm0 = da_gtsm0.vector.clip_bbox(bbox)

da_gtsm0.raster.set_crs(4326)
gdf_gtsm = da_gtsm0.vector.to_gdf()
mdt0 = da_mdt.raster.sample(gdf_gtsm).rename({'index': 'stations'}).reset_coords(drop=True)
da_gtsm0.station_name.values

In [ ]:
gdf_pits1, cmf_gtsm_ref = cmf_pit_mapping(gdf_gtsm, name)
fig, ax = plt.subplots(1,1, figsize=(6,6))
gdf_pits.plot(markersize=(gdf_pits['upa_10'].values-7)*7, color='b', ax=ax)
gdf_pits1.plot(markersize=(gdf_pits1['upa_10'].values-7)*7, color='k', ax=ax)
gdf_gtsm.iloc[np.unique(gdf_pits1['gtsm_idx'].values-1)].plot(markersize=15, color='r', marker='^', ax=ax)
cmf_gtsm_ref

In [ ]:
for run in runs:
    fn = join(root, name, f'{prefix}_{run}_his.nc')
    fn_out = fn.replace('.nc', '_beira.nc')

    # READ
    da_gtsm = xr.open_dataset(fn, chunks=chunks)[var].sel(stations=gdf_gtsm.index)

    # reindex to get timeseries from JAN-01
    time = xr.IndexVariable('time', pd.date_range(tstart, tstop, freq='10MIN'))
    da_gtsm = da_gtsm.resample(time='10MIN').nearest().reindex(time=time, fill_value=0)

    # apply MDT
    if 'tides' in run:
        da_gtsm = da_gtsm + mdt0
        fn_out = fn_out.replace('.nc', '_egm.nc')

    # write to nc
    da_gtsm.name = var
    encoding = {var: {'dtype': 'float32'}}
    da_gtsm.to_netcdf(fn_out, encoding=encoding)

In [ ]:
da_gtsm

In [ ]:
da_gtsm.sel(time=slice('2021-01-18', '2021-01-25')).plot.line(x='time')